# Smart Public Transport Operations — MTC Chennai Predictive Analytics
## Operational Problem Statement: Predicting Transit Overcrowding, Imbalance, and Dispatch Interventions
---
**Target Users:** CUMTA (Chennai Unified Metropolitan Transport Authority) & MTC (Metropolitan Transport Corporation) Central Control Room Operators  
**Dataset:** 31 Days of High-Frequency Electronic Ticketing Machine (ETM) Records (March 1 – March 31, 2026), 67.9 Million raw ticket transactions across 705 bus routes and 2,717 geocoded bus stops in Chennai Metropolitan Area.

## 1. Project Architecture & Pipeline Overview
The pipeline ingests raw ETM ticketing logs, transforms them into high-performance columnar Parquet tables, extracts route-hour demand and capacity profiles, trains machine learning models to forecast next-hour demand and overcrowding risks, and translates predictions into automated dispatch recommendations for control room staff.

```
[ 67.9M Raw ETM Tickets (31 Days) ]
                 │
                 ▼ Phase 1 (Parallel Streaming Ingestion)
[ 31 Clean Daily Parquets + 2,717 Geocoded Stops ]
                 │
                 ▼ Phase 2 (Capacity & Imbalance Modeling)
[ 349,799 Route-Hour Slots | MoHUA Bus Capacity Benchmarks | Isolation Forest Anomalies ]
                 │
                 ▼ Phase 3 (Machine Learning Predictive Models)
[ XGBoost Regressor (R²=0.9637) & XGBoost Overcrowding Classifier (AUC=0.9252) ]
                 │
                 ▼ Phase 4 (Actionable Recommendation Engine)
[ 6-Field Action Engine: Reserve Injection | Short-Turn | Surplus Reallocation | Rail Divert ]
                 │
                 ▼ Phase 5 (Deliverables & Dashboard Feed)
[ Control Room JSON Feed | Interactive Notebook | outputs/findings.md ]
```

In [ ]:
# Environment setup and imports
import os, sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

sns.set_theme(style='whitegrid', font='sans-serif')
ROOT = Path('.')
print('Workspace initialized. Ready to inspect models and operational metrics.')

## 2. Phase 1: Data Ingestion & Hygiene
We processed 31 consecutive compressed CSV files (`Ticket-Data-2026-03-01.csv.bz2` to `Ticket-Data-2026-03-31.csv.bz2`), validating 36 ETM telemetry columns.

In [ ]:
# Display Phase 1 Daily Summary Table
daily_summary = pd.read_csv('outputs/tables/phase1_daily_summary.csv')
display(daily_summary.head(10))

print(f"Total Clean Passenger Journeys: {daily_summary['passengers'].sum():,}")
print(f"Total Fare Revenue: ₹{daily_summary['revenue_inr'].sum():,.2f}")
print(f"Average Daily Weekday Passengers: {daily_summary[~daily_summary['is_weekend']]['passengers'].mean():,.0f}")
print(f"Average Daily Weekend Passengers: {daily_summary[daily_summary['is_weekend']]['passengers'].mean():,.0f}")

### Phase 1 Visualizations: Daily Ridership & Diurnal Demand Curves

In [ ]:
display(Image('outputs/charts/phase1_daily_summary.png'))
display(Image('outputs/charts/phase1_hourly_distribution.png'))

## 3. Phase 2: Exploratory Operational Analysis (Demand vs. Supply)
Using MoHUA / CUMTA service benchmarks (70 passengers rated normal capacity: 40 seated + 30 standees, 85 crush limit), we computed hourly load factors across 349,799 route-hour slots.

In [ ]:
# Top Overcrowded Corridors in Chennai
top_crowded = pd.read_csv('outputs/tables/phase2_top_overcrowded_routes.csv')
print("TOP 10 CHRONICALLY OVERCROWDED CORRIDORS:")
display(top_crowded.head(10))

# Top Supply-Demand Imbalances (Deficits vs Surpluses)
imbalances = pd.read_csv('outputs/tables/phase2_top_imbalance_slots.csv')
print("CRITICAL SUPPLY DEFICITS (Severe Bottlenecks):")
display(imbalances[imbalances['type'] == 'DEFICIT'].head(5))
print("CRITICAL SUPPLY SURPLUSES (Fleet Inefficiencies):")
display(imbalances[imbalances['type'] == 'SURPLUS'].head(5))

### Phase 2 Visualizations: Load Factor Distribution & Imbalance Matrix

In [ ]:
display(Image('outputs/charts/phase2_load_factor_distribution.png'))
display(Image('outputs/charts/phase2_top_deficits_surpluses.png'))

## 4. Phase 3: Machine Learning Prediction Models
We trained two machine learning models evaluated on a strict chronological out-of-time test set (March 27–31, 2026):
1. **XGBoost Regressor:** Predicts next-hour passenger ridership ($D_{r, t+1}$).
2. **XGBoost Classifier:** Predicts next-hour overcrowding probability ($P(\text{Load Factor} > 1.0)$).

Both models are benchmarked directly against the official baseline: **Previous-Week Lag Persistence Baseline ($t - 168\text{h}$)**.

In [ ]:
metrics_df = pd.read_csv('outputs/tables/phase3_model_metrics.csv')
display(metrics_df)

feat_imp = pd.read_csv('outputs/tables/phase3_feature_importance.csv')
print("TOP 5 PREDICTIVE FEATURES (REGRESSION GAIN):")
display(feat_imp.sort_values('reg_gain', ascending=False).head(5))

### Phase 3 Visualizations: Model Calibration, Error Distributions & ROC Curve

In [ ]:
display(Image('outputs/charts/phase3_actual_vs_predicted.png'))
display(Image('outputs/charts/phase3_roc_curve.png'))
display(Image('outputs/charts/phase3_feature_importance.png'))
display(Image('outputs/charts/phase3_hourly_error_profile.png'))

## 5. Phase 4: Operational Recommendation Engine
Our rule-based operational engine translates ML model forecasts into the 6 mandatory operational fields required by CUMTA / MTC control room dispatchers:
1. **Location:** Route corridor & associated depot.
2. **Time Window:** Target forecast hour.
3. **Predicted Problem:** Severity of crush or under-utilization.
4. **Recommended Action:** Concrete dispatch intervention.
5. **Operational Reason:** Data-driven justification.
6. **Expected Impact:** Quantified load factor reduction or operational cost savings.

In [ ]:
recs_table = pd.read_csv('outputs/tables/recommendations.csv')
print("TOP 10 ACTIONABLE CONTROL ROOM RECOMMENDATIONS:")
display(recs_table.head(10))

### Phase 4 Visualizations: Action Categories & Load Factor Normalization

In [ ]:
display(Image('outputs/charts/phase4_recommendations_action_breakdown.png'))
display(Image('outputs/charts/phase4_impact_simulation.png'))

## 6. Real-Time Control Room JSON Dashboard Feed
To enable instantaneous integration with CUMTA's Computer-Aided Dispatch (CAD) and Automatic Vehicle Location (AVL) dashboards, the pipeline streams a JSON feed containing per-route risk scores, predicted load factors, and prioritized alert popups.

In [ ]:
with open('outputs/tables/dashboard_feed.json') as f:
    feed = json.load(f)

print("FEED METADATA:")
print(json.dumps(feed['system_metadata'], indent=2))

print("\nSAMPLE ACTIVE ROUTE TELEMETRY IN FEED (First 3 Routes):")
print(json.dumps(feed['active_routes_status'][:3], indent=2))

## 7. Conclusions & Strategic Policy Recommendations for CUMTA
1. **ETM Telemetry is Production-Ready for Predictive Dispatch:** Transmitting lightweight ticket batches (<15 KB/sec citywide) provides 100x cheaper, more resilient crowd intelligence than onboard CCTV video streaming.
2. **Dynamic Fleet Swapping Cuts ₹Millions in Wasted Fuel:** Over 4,900 route-hours feature empty bus surpluses (Load Factor < 35%) while parallel corridors face severe crush (>125%). Reallocating 2-3 buses from surplus to deficit corridors restores service balance with zero net fleet acquisition costs.
3. **Multi-Modal Integration (Bus + Rail):** Peak bus crush on corridors like GST Road (Route 70V, 21G) and OMR (Route 570, 102) can be rapidly alleviated by broadcasting CUMTA multi-modal dynamic advisories redirecting passengers to the Suburban Rail and Metro networks.